# KOA Decision-Domain Evaluation — EGS + Neural Complementation + DGS

## Objective

This notebook evaluates the knowledge produced by the proposed methodology without embedding domain-specific decision semantics in the evaluation code.

The experiment is divided into two phases.

### Phase 1 — Formal Grounding and Symbolic Decision

1. The notebook loads the `KOA_semantic_bridge.pl` artifact generated by the methodology.
2. The assessment dimension, decision criteria, criterion targets, semantic signatures, and decision-level rules are discovered dynamically from the Semantic Bridge Model.
3. Gemini evaluates the available organizational evidence for every criterion defined by the selected Decision Domain, assigning one epistemic state: `S/P/U/C`.
4. The **Evidence Grounding Score (EGS)** is computed exclusively from criteria in state `S`.
5. The formal decision level is derived deterministically from `level_rule/3` facts in the Semantic Bridge Model.
6. The neural component does **not** assign the formal decision level.

### Phase 2 — Neural Complementation

1. Only criteria classified as `P` or `U` in Phase 1 are submitted to neural complementation.
2. For each unresolved criterion, Gemini receives the criterion definition obtained from the Semantic Bridge Model, the previous formal assessment, and the organizational evidence available in the contract and linked technical specification, when present.
3. The neural result is one of `Supported`, `Contradicted`, or `Still Undetermined`.
4. The original formal state `S/P/U/C` is preserved and never overwritten.
5. The **Decision Grounding Score (DGS)** measures how much of the criterion set can be conclusively assessed after neural complementation.
6. The symbolic decision remains authoritative and is preserved separately.
7. A complementary neuro-symbolic recommendation is computed using the same decision rules, while allowing `P/U` criteria assessed as `Supported` by the neural component.

The notebook therefore separates:

\[
EGS \rightarrow \text{positive formal grounding}
\]

\[
DGS_{\mathrm{symbolic}} \rightarrow \text{formal evidential determination}
\]

\[
DGS_{\mathrm{neuro-symbolic}} \rightarrow \text{decision determination after neural complementation}
\]

and

\[
\Delta DGS =
DGS_{\mathrm{neuro-symbolic}} -
DGS_{\mathrm{symbolic}}
\]

as a descriptive measure of the additional determination provided by neural complementation.

### Experimental principle

The evaluation code may define **how** knowledge is evaluated, but it must not contain the **Decision Domain knowledge being evaluated**. Decision criteria, semantic requirements, and decision-level rules are therefore read from the Semantic Bridge Model rather than hard-coded in Python or in the prompts.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Imports and Gemini client configuration.
import os
import json
import re
import time
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from google import genai

try:
    from google.colab import userdata
    if not os.environ.get("GEMINI_API_KEY"):
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GEMINI_API_KEY"] = key
except Exception:
    pass

model_name = "gemini-3.5-flash-lite"
client = genai.Client(
    api_key=os.environ.get("GEMINI_API_KEY")
    or os.environ.get("GOOGLE_API_KEY")
)


In [ ]:
# Configure repository paths.
# Adjust these two folders to match the execution environment.
directory = '/content/drive/My Drive/KOA/onboarding/contratos_COM_UFO/'
persist_directory = '/content/drive/My Drive/KOA/onboarding/modelos/'

PL_FOLDER = directory
SEMANTIC_BRIDGE_PATH = str(Path(persist_directory) / "KOA_semantic_bridge.pl")


## Loading organizational artifacts and the Semantic Bridge Model

- Each contract is represented as a Prolog `.pl` file.
- A linked Technical Specification (`_ET.pl`) is loaded when available.
- `KOA_semantic_bridge.pl` contains the formalized Decision Domain, decision criteria, semantic signatures, and executable decision-level rules used by the evaluation.


In [ ]:
# Load contract Prolog files as text attachments.
@dataclass
class AttachmentText:
    filename: str
    text: str
    et_filename: Optional[str] = None
    et_text: Optional[str] = None

def derive_et_filename(contract_filename: str) -> str:
    stem, ext = os.path.splitext(contract_filename)
    return f"{stem}_ET{ext}"

def list_directory_files(folder: str) -> List[str]:
    p = Path(folder)
    return sorted([fp.name for fp in p.iterdir() if fp.is_file()], key=str.lower)

def build_directory_file_index(folder: str) -> Dict[str, Path]:
    p = Path(folder)
    return {fp.name.lower(): fp for fp in p.iterdir() if fp.is_file()}

def load_pl_attachments(folder: str, pattern: str = "KOA_UFO_contrato_ocs_*.pl") -> List[AttachmentText]:
    p = Path(folder)
    attachments: List[AttachmentText] = []
    for fp in sorted(p.glob(pattern), key=lambda x: x.name.lower()):
        # Technical Specification files are not loaded as primary contracts.
        if fp.stem.endswith("_ET"):
            continue

        content = fp.read_text(encoding="utf-8", errors="replace")

        et_filename = derive_et_filename(fp.name)
        et_path = fp.with_name(et_filename)
        et_text = None
        if et_path.exists():
            et_text = et_path.read_text(encoding="utf-8", errors="replace")

        attachments.append(
            AttachmentText(
                filename=fp.name,
                text=content,
                et_filename=et_filename if et_text is not None else None,
                et_text=et_text,
            )
        )
    return attachments

DIRECTORY_FILES = list_directory_files(PL_FOLDER)
DIRECTORY_FILE_INDEX = build_directory_file_index(PL_FOLDER)
ATTACHMENTS = load_pl_attachments(PL_FOLDER)

print("Loaded contract .pl files:", len(ATTACHMENTS))
print("All files found in directory:", len(DIRECTORY_FILES))
num_with_et = sum(1 for a in ATTACHMENTS if a.et_text)
print("Contracts with matching ET:", num_with_et)
if not ATTACHMENTS:
    raise RuntimeError(f"No contract .pl files found in: {PL_FOLDER}")


In [ ]:
# Load the Semantic Bridge Model as Prolog text.
def read_pl_as_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

SEMANTIC_BRIDGE_PL = read_pl_as_text(SEMANTIC_BRIDGE_PATH)


In [ ]:
# Render the organizational evidence supplied to the model.
def render_contract_context(attachment: AttachmentText) -> str:
    contract_block = f"""
CONTRACT FILE: {attachment.filename}
---
{attachment.text}
---
"""

    if attachment.et_text:
        et_block = f"""
TECHNICAL SPECIFICATION FILE: {attachment.et_filename}
---
{attachment.et_text}
---
"""
    else:
        et_block = """
TECHNICAL SPECIFICATION FILE:
---
No linked Technical Specification was provided for this contract.
---
"""

    return contract_block + "\n" + et_block


## Phase 1 — Batch evidence-grounding evaluation

This section evaluates the assessment dimension defined in the Semantic Bridge Model for every loaded contract.

The criterion set is **not hard-coded**. The notebook discovers all criteria associated with the selected assessment dimension through `assessment_dimension/1` and `dimension_criterion/2`. Criterion targets, operationalizing questions, semantic signatures, and decision-level rules are also read from `KOA_semantic_bridge.pl`.

The neural component evaluates only the evidential state `S/P/U/C` of the dynamically discovered criteria. The final decision level is **not requested from the model**. It is derived deterministically from the executable `level_rule/3` facts in the Semantic Bridge Model.

EGS is computed independently from the final decision level.


In [ ]:
# ==============================
# Phase 1 configuration
# ==============================
from pathlib import Path
from datetime import datetime
import json
import re
import time
import pandas as pd

BATCH_PERSONA = "tecnico"
BATCH_LIMIT = None
SLEEP_SECONDS = 10.0

EVALUATION_OUTPUT_DIR = Path(PL_FOLDER) / "evaluation_outputs"
EVALUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_STATES = {"S", "P", "U", "C"}


# ----------------------------------------------------------------------
# Semantic Bridge Model parsing
# ----------------------------------------------------------------------

def _unquote_prolog_string(value: str) -> str:
    value = (value or "").strip()
    if len(value) >= 2 and value[0] == '"' and value[-1] == '"':
        value = value[1:-1]
    return value.replace('\\"', '"').replace("\\n", "\n")


def _parse_atom_list(raw: str):
    return [
        item.strip()
        for item in (raw or "").split(",")
        if item.strip()
    ]


def parse_semantic_bridge_model(bridge_text: str):
    """
    Parses the decision structures required by the evaluation directly from
    KOA_semantic_bridge.pl.

    No Decision Domain criterion, semantic signature, or decision-level rule
    is defined in Python.
    """
    text = bridge_text or ""

    dimensions = re.findall(
        r"assessment_dimension\s*\(\s*([a-zA-Z0-9_]+)\s*\)\s*\.",
        text,
        flags=re.IGNORECASE,
    )
    dimensions = [d.lower() for d in dimensions]

    dimension_criteria = {}
    for dim, cid in re.findall(
        r"dimension_criterion\s*\(\s*([a-zA-Z0-9_]+)\s*,\s*([a-zA-Z0-9_]+)\s*\)\s*\.",
        text,
        flags=re.IGNORECASE,
    ):
        dimension_criteria.setdefault(dim.lower(), []).append(cid.lower())

    criterion_targets = {
        cid.lower(): target.lower()
        for cid, target in re.findall(
            r"criterion_target\s*\(\s*([a-zA-Z0-9_]+)\s*,\s*([a-zA-Z0-9_]+)\s*\)\s*\.",
            text,
            flags=re.IGNORECASE,
        )
    }

    criterion_concepts = {
        cid.lower(): concept.lower()
        for cid, concept in re.findall(
            r"criterion_concept\s*\(\s*([a-zA-Z0-9_]+)\s*,\s*([a-zA-Z0-9_]+)\s*\)\s*\.",
            text,
            flags=re.IGNORECASE,
        )
    }

    concept_labels = {
        cid.lower(): _unquote_prolog_string(label)
        for cid, label in re.findall(
            r'concept_label\s*\(\s*([a-zA-Z0-9_]+)\s*,\s*("(?:[^"\\]|\\.)*")\s*\)\s*\.',
            text,
            flags=re.IGNORECASE,
        )
    }

    frame_questions = {
        qf.lower(): _unquote_prolog_string(question)
        for qf, question in re.findall(
            r'frame_question\s*\(\s*([a-zA-Z0-9_]+)\s*,\s*("(?:[^"\\]|\\.)*")\s*\)\s*\.',
            text,
            flags=re.IGNORECASE,
        )
    }

    criterion_questions = {}
    for qf, cid in re.findall(
        r"question_operationalizes_criterion\s*\(\s*([a-zA-Z0-9_]+)\s*,\s*([a-zA-Z0-9_]+)\s*\)\s*\.",
        text,
        flags=re.IGNORECASE,
    ):
        qf = qf.lower()
        cid = cid.lower()
        if qf in frame_questions:
            criterion_questions[cid] = frame_questions[qf]

    criterion_signatures = {
        cid.lower(): sig.lower()
        for cid, sig in re.findall(
            r"criterion_signature\s*\(\s*([a-zA-Z0-9_]+)\s*,\s*([a-zA-Z0-9_]+)\s*\)\s*\.",
            text,
            flags=re.IGNORECASE,
        )
    }

    signature_requirements = {}
    for sig, requirement in re.findall(
        r"sig_requires\s*\(\s*([a-zA-Z0-9_]+)\s*,\s*(.*?)\)\s*\.",
        text,
        flags=re.IGNORECASE,
    ):
        signature_requirements.setdefault(sig.lower(), []).append(
            requirement.strip()
        )

    dimension_levels = {}
    level_order_in_file = {}
    for dim, level in re.findall(
        r"dimension_level\s*\(\s*([a-zA-Z0-9_]+)\s*,\s*([a-zA-Z0-9_]+)\s*\)\s*\.",
        text,
        flags=re.IGNORECASE,
    ):
        dim = dim.lower()
        level = level.lower()
        dimension_levels.setdefault(dim, []).append(level)
        level_order_in_file[(dim, level)] = len(dimension_levels[dim])

    level_rules = {}
    for dim, level, raw_criteria in re.findall(
        r"level_rule\s*\(\s*([a-zA-Z0-9_]+)\s*,\s*([a-zA-Z0-9_]+)\s*,\s*\[([^\]]*)\]\s*\)\s*\.",
        text,
        flags=re.IGNORECASE | re.MULTILINE,
    ):
        level_rules[(dim.lower(), level.lower())] = [
            c.lower() for c in _parse_atom_list(raw_criteria)
        ]

    level_concepts = {
        (dim.lower(), level.lower()): concept.lower()
        for dim, level, concept in re.findall(
            r"level_concept\s*\(\s*([a-zA-Z0-9_]+)\s*,\s*([a-zA-Z0-9_]+)\s*,\s*([a-zA-Z0-9_]+)\s*\)\s*\.",
            text,
            flags=re.IGNORECASE,
        )
    }

    level_descriptions = {
        (dim.lower(), level.lower()): _unquote_prolog_string(desc)
        for dim, level, desc in re.findall(
            r'level_description\s*\(\s*([a-zA-Z0-9_]+)\s*,\s*([a-zA-Z0-9_]+)\s*,\s*("(?:[^"\\]|\\.)*")\s*\)\s*\.',
            text,
            flags=re.IGNORECASE,
        )
    }

    # If category_level/5 contains the same level atoms, its numeric position
    # is used as an explicit rank. Otherwise, source order is retained.
    level_rank_candidates = {}
    for level, rank in re.findall(
        r'category_level\s*\(\s*[a-zA-Z0-9_]+\s*,\s*([a-zA-Z0-9_]+)\s*,\s*(\d+)\s*,',
        text,
        flags=re.IGNORECASE,
    ):
        level_rank_candidates.setdefault(level.lower(), []).append(int(rank))

    return {
        "dimensions": dimensions,
        "dimension_criteria": dimension_criteria,
        "criterion_targets": criterion_targets,
        "criterion_concepts": criterion_concepts,
        "concept_labels": concept_labels,
        "criterion_questions": criterion_questions,
        "criterion_signatures": criterion_signatures,
        "signature_requirements": signature_requirements,
        "dimension_levels": dimension_levels,
        "level_rules": level_rules,
        "level_concepts": level_concepts,
        "level_descriptions": level_descriptions,
        "level_order_in_file": level_order_in_file,
        "level_rank_candidates": level_rank_candidates,
    }


SEMANTIC_BRIDGE_MODEL = parse_semantic_bridge_model(SEMANTIC_BRIDGE_PL)

# The evaluation is generic when the bridge contains one assessment dimension.
# If a future Decision Domain contains more than one dimension, set this value
# explicitly to one of the dimensions discovered from the bridge.
ASSESSMENT_DIMENSION = None

if ASSESSMENT_DIMENSION is None:
    available_dimensions = SEMANTIC_BRIDGE_MODEL["dimensions"]
    if len(available_dimensions) != 1:
        raise ValueError(
            "ASSESSMENT_DIMENSION must be specified when the Semantic Bridge "
            f"Model contains {len(available_dimensions)} dimensions: "
            f"{available_dimensions}"
        )
    ASSESSMENT_DIMENSION = available_dimensions[0]
else:
    ASSESSMENT_DIMENSION = ASSESSMENT_DIMENSION.lower()

if ASSESSMENT_DIMENSION not in SEMANTIC_BRIDGE_MODEL["dimension_criteria"]:
    raise ValueError(
        f"Assessment dimension '{ASSESSMENT_DIMENSION}' is not defined in "
        "the Semantic Bridge Model."
    )


def criterion_descriptor(criterion_id: str):
    cid = criterion_id.lower()
    model = SEMANTIC_BRIDGE_MODEL

    target = model["criterion_targets"].get(cid, "")
    concept_id = model["criterion_concepts"].get(cid)
    concept_label = model["concept_labels"].get(concept_id, "") if concept_id else ""
    question = model["criterion_questions"].get(cid, "")
    signature = model["criterion_signatures"].get(cid)
    requirements = model["signature_requirements"].get(signature, []) if signature else []

    label = concept_label or target.replace("_", " ").strip() or cid

    return {
        "criterion_id": cid,
        "criterion_label": label,
        "target": target,
        "operationalizing_question": question,
        "signature_id": signature or "",
        "signature_requirements": requirements,
    }


DECISION_CRITERIA = [
    criterion_descriptor(cid)
    for cid in SEMANTIC_BRIDGE_MODEL["dimension_criteria"][ASSESSMENT_DIMENSION]
]
DECISION_CRITERION_IDS = [c["criterion_id"] for c in DECISION_CRITERIA]
DECISION_CRITERION_BY_ID = {
    c["criterion_id"]: c for c in DECISION_CRITERIA
}


def _level_rank(level: str):
    candidates = SEMANTIC_BRIDGE_MODEL["level_rank_candidates"].get(level, [])
    if len(set(candidates)) == 1:
        return candidates[0]
    return SEMANTIC_BRIDGE_MODEL["level_order_in_file"].get(
        (ASSESSMENT_DIMENSION, level),
        0,
    )


DECISION_LEVELS = sorted(
    SEMANTIC_BRIDGE_MODEL["dimension_levels"].get(ASSESSMENT_DIMENSION, []),
    key=_level_rank,
)

DECISION_LEVEL_RULES = {
    level: SEMANTIC_BRIDGE_MODEL["level_rules"].get(
        (ASSESSMENT_DIMENSION, level),
        [],
    )
    for level in DECISION_LEVELS
}


def level_display_label(level: str):
    concept_id = SEMANTIC_BRIDGE_MODEL["level_concepts"].get(
        (ASSESSMENT_DIMENSION, level)
    )
    if concept_id:
        label = SEMANTIC_BRIDGE_MODEL["concept_labels"].get(concept_id)
        if label:
            return label
    return level.replace("_", " ")


DECISION_LEVEL_LABELS = {
    level: level_display_label(level)
    for level in DECISION_LEVELS
}

DEFAULT_LEVELS = [
    level for level, required in DECISION_LEVEL_RULES.items()
    if not required
]
if not DEFAULT_LEVELS:
    raise ValueError(
        "The selected assessment dimension has no default level_rule/3 "
        "with an empty criterion list."
    )


def build_decision_domain_context():
    lines = [
        f"Assessment dimension: {ASSESSMENT_DIMENSION}",
        "Decision criteria discovered from the Semantic Bridge Model:",
    ]
    for criterion in DECISION_CRITERIA:
        lines.append(f"- id: {criterion['criterion_id']}")
        if criterion["target"]:
            lines.append(f"  target: {criterion['target']}")
        if criterion["operationalizing_question"]:
            lines.append(
                "  operationalizing question: "
                + criterion["operationalizing_question"]
            )
        if criterion["signature_id"]:
            lines.append(
                f"  semantic signature: {criterion['signature_id']}"
            )
        if criterion["signature_requirements"]:
            lines.append(
                "  signature requirements: "
                + "; ".join(criterion["signature_requirements"])
            )
    return "\n".join(lines)


DECISION_DOMAIN_CONTEXT = build_decision_domain_context()


def normalize_decision_criteria(conditions):
    """
    Ensures that the result contains exactly the criteria defined for the
    selected assessment dimension in the Semantic Bridge Model.
    """
    by_id = {}

    for cond in conditions or []:
        cid = str(cond.get("criterion_id", "")).strip().lower()
        if cid not in DECISION_CRITERION_BY_ID or cid in by_id:
            continue

        state = str(cond.get("state", "U")).strip().upper()
        if state not in VALID_STATES:
            state = "U"

        descriptor = DECISION_CRITERION_BY_ID[cid]
        normalized = dict(cond)
        normalized["criterion_id"] = cid
        normalized["criterion_label"] = descriptor["criterion_label"]
        normalized["criterion_target"] = descriptor["target"]
        normalized["state"] = state
        normalized.setdefault("evidence", "")
        normalized.setdefault("rationale", "")
        by_id[cid] = normalized

    output = []
    for descriptor in DECISION_CRITERIA:
        cid = descriptor["criterion_id"]
        if cid in by_id:
            output.append(by_id[cid])
        else:
            output.append({
                "criterion_id": cid,
                "criterion_label": descriptor["criterion_label"],
                "criterion_target": descriptor["target"],
                "state": "U",
                "evidence": "",
                "rationale": (
                    "The criterion was defined by the Semantic Bridge Model "
                    "but was not returned by the model; it was therefore "
                    "normalized to Undetermined."
                ),
            })

    return output


def compute_evidence_grounding_score(conditions):
    """
    EGS(d) = number of formally satisfied criteria / number of criteria
    defined for the selected assessment dimension.
    """
    conditions = normalize_decision_criteria(conditions)
    states = [c["state"] for c in conditions]
    counts = {s: states.count(s) for s in sorted(VALID_STATES)}
    counts["N"] = len(DECISION_CRITERIA)

    grounded = counts.get("S", 0)
    denominator = len(DECISION_CRITERIA)
    egs = grounded / denominator if denominator else 0.0

    return round(egs, 4), counts, conditions


def derive_decision_from_positive_criteria(positive_criteria):
    """
    Applies the executable level_rule/3 semantics from the Semantic Bridge
    Model. If several levels match, the highest-ranked level wins.
    """
    positive = set(positive_criteria or [])

    matched_levels = []
    for level in DECISION_LEVELS:
        required = set(DECISION_LEVEL_RULES[level])
        if required.issubset(positive):
            matched_levels.append(level)

    if not matched_levels:
        raise ValueError(
            "No decision level matches the supplied criteria and no usable "
            "default rule was found."
        )

    winning_level = max(matched_levels, key=_level_rank)

    return {
        "decision_label": DECISION_LEVEL_LABELS[winning_level],
        "symbolic_level": winning_level,
        "positive_criteria": sorted(positive),
        "matched_levels": matched_levels,
        "winning_rule": DECISION_LEVEL_RULES[winning_level],
    }


def derive_symbolic_decision(conditions):
    conditions = normalize_decision_criteria(conditions)
    satisfied = {
        c["criterion_id"]
        for c in conditions
        if c["state"] == "S"
    }
    return derive_decision_from_positive_criteria(satisfied)


def build_symbolic_rationale(symbolic_result, parse_error=False):
    final = symbolic_result["decision_label"]
    satisfied = symbolic_result["positive_criteria"]
    winning = symbolic_result["winning_rule"]

    if parse_error:
        return (
            f"Decision level '{final}' was derived only from the executable "
            "rules in the Semantic Bridge Model. The neural response could "
            "not be parsed as structured JSON, so missing criterion states "
            "were normalized to U. The technical failure remains recorded "
            "in parse_error=True."
        )

    if winning:
        return (
            f"Decision level '{final}' was derived by symbolic inference. "
            f"The winning rule requires {winning}, and all required criteria "
            f"were formally established as S. Satisfied criteria: {satisfied}."
        )

    return (
        f"Decision level '{final}' was derived by symbolic inference from "
        "the default empty rule because no higher-ranked rule had all of "
        f"its required criteria formally established as S. "
        f"Satisfied criteria: {satisfied}."
    )


def extract_contract_label(filename: str) -> str:
    m = re.search(
        r'ocs[_\s-]?(\d{1,4})[_/-](\d{4})',
        filename or '',
        flags=re.IGNORECASE,
    )
    if m:
        return f"OCS {int(m.group(1))}/{m.group(2)}"
    return filename or ""


def build_phase1_json_schema():
    condition_objects = []
    for criterion in DECISION_CRITERIA:
        condition_objects.append({
            "criterion_id": criterion["criterion_id"],
            "state": "S|P|U|C",
            "evidence": "short evidence from the organizational artifacts",
            "rationale": "brief justification for the assigned state",
        })

    return json.dumps(
        {
            "contract_file": "<contract filename>",
            "conditions": condition_objects,
            "evidence_summary": "brief summary of the evidence assessment",
        },
        ensure_ascii=False,
        indent=2,
    )


def build_evidence_grounding_prompt(attachment: AttachmentText) -> str:
    contract_label = extract_contract_label(attachment.filename)

    return f"""
You are performing a FORMAL EVIDENCE-GROUNDING ASSESSMENT for one
organizational contract.

The Decision Domain semantics have already been formalized in the supplied
Semantic Bridge Model. You MUST NOT introduce new decision criteria, redefine
criterion meaning, or use hidden domain rules.

Your task is limited to determining how the organizational evidence relates
to every criterion associated with the selected assessment dimension.

Assessment dimension:
{ASSESSMENT_DIMENSION}

Decision Domain structures parsed from the Semantic Bridge Model:
---
{DECISION_DOMAIN_CONTEXT}
---

Epistemic states:
- S = Satisfied: explicit organizational evidence establishes the criterion's
  truthmaker conditions.
- P = Partially Satisfied: relevant evidence exists, but some required meaning
  remains incomplete or indirect.
- U = Undetermined: the available organizational artifacts do not provide
  enough evidence to establish or contradict the criterion.
- C = Contradicted: explicit organizational evidence establishes that the
  criterion does not hold.

Grounding rules:
1. Evaluate semantic equivalence, not lexical identity.
2. The Semantic Bridge Model defines criterion meaning and semantic
   requirements; it is NOT contract-specific evidence.
3. Contract-specific evidence must come from the supplied contract and linked
   Technical Specification, when available.
4. Do not use general domain plausibility to assign S.
5. Absence of evidence is U, not C.
6. Use C only when explicit organizational evidence supports contradiction.
7. Do not infer or return the final decision level. The final decision is
   computed later by symbolic execution of level_rule/3.
8. Return exactly one condition object for every criterion listed above.
9. Do not add criteria that are not defined by the Semantic Bridge Model.
10. Do not omit a criterion. If evidence is insufficient, return U.

Return STRICT JSON only, with no markdown fences.

Contract:
- file: {attachment.filename}
- label: {contract_label}

Expected JSON structure:
{build_phase1_json_schema()}

SEMANTIC BRIDGE MODEL:
---
{SEMANTIC_BRIDGE_PL}
---

ORGANIZATIONAL EVIDENCE:
---
{render_contract_context(attachment)}
---

Return JSON only.
""".strip()


def _extract_json_object(text: str):
    if not text:
        return None

    cleaned = text.strip()
    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        return json.loads(cleaned)
    except Exception:
        pass

    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start >= 0 and end > start:
        try:
            return json.loads(cleaned[start:end + 1])
        except Exception:
            return None

    return None


def evaluate_contract(attachment: AttachmentText, sleep_seconds: float = 0.0):
    prompt = build_evidence_grounding_prompt(attachment)
    response = client.models.generate_content(
        model=model_name,
        contents=prompt,
    )
    raw_text = getattr(response, "text", "").strip()
    parsed = _extract_json_object(raw_text)

    if parsed is None:
        parsed = {
            "contract_file": attachment.filename,
            "contract_label": extract_contract_label(attachment.filename),
            "assessment_dimension": ASSESSMENT_DIMENSION,
            "conditions": [],
            "evidence_summary": (
                "The model response could not be parsed as structured JSON."
            ),
            "parse_error": True,
            "raw_response": raw_text,
        }
    else:
        parsed["parse_error"] = False
        parsed["raw_response"] = raw_text
        parsed["contract_file"] = attachment.filename
        parsed["contract_label"] = extract_contract_label(
            attachment.filename
        )
        parsed["assessment_dimension"] = ASSESSMENT_DIMENSION

    egs, state_counts, normalized_conditions = (
        compute_evidence_grounding_score(parsed.get("conditions", []))
    )
    parsed["conditions"] = normalized_conditions
    parsed["evidence_grounding_score"] = egs
    parsed["state_counts"] = state_counts

    symbolic = derive_symbolic_decision(normalized_conditions)
    parsed["final_decision"] = symbolic["decision_label"]
    parsed["symbolic_inference"] = {
        "symbolic_level": symbolic["symbolic_level"],
        "satisfied_criteria": symbolic["positive_criteria"],
        "matched_levels": symbolic["matched_levels"],
        "winning_rule": symbolic["winning_rule"],
    }
    parsed["symbolic_rationale"] = build_symbolic_rationale(
        symbolic,
        parse_error=parsed["parse_error"],
    )

    if sleep_seconds and sleep_seconds > 0:
        time.sleep(sleep_seconds)

    return parsed


def flatten_result_for_table(result: dict):
    counts = result.get("state_counts", {}) or {}
    return {
        "contract_file": result.get("contract_file"),
        "contract_label": result.get("contract_label"),
        "assessment_dimension": result.get("assessment_dimension"),
        "final_decision": result.get("final_decision"),
        "evidence_grounding_score": result.get("evidence_grounding_score"),
        "S": counts.get("S", 0),
        "P": counts.get("P", 0),
        "U": counts.get("U", 0),
        "C": counts.get("C", 0),
        "N": counts.get("N", len(DECISION_CRITERIA)),
        "parse_error": result.get("parse_error", False),
        "symbolic_rationale": result.get("symbolic_rationale", ""),
    }


def compute_criterion_egs(results):
    total_contracts = len(results)
    rows = []

    for criterion in DECISION_CRITERIA:
        cid = criterion["criterion_id"]
        grounded = 0
        state_counts = {s: 0 for s in sorted(VALID_STATES)}

        for result in results:
            conditions = normalize_decision_criteria(
                result.get("conditions", [])
            )
            cond = next(
                c for c in conditions
                if c["criterion_id"] == cid
            )
            state = cond["state"]
            state_counts[state] += 1
            if state == "S":
                grounded += 1

        egs = grounded / total_contracts if total_contracts else 0.0

        rows.append({
            "criterion_id": cid,
            "criterion_label": criterion["criterion_label"],
            "criterion_target": criterion["target"],
            "grounded_contracts": grounded,
            "total_contracts": total_contracts,
            "evidence_grounding_score": round(egs, 4),
            "S": state_counts["S"],
            "P": state_counts["P"],
            "U": state_counts["U"],
            "C": state_counts["C"],
        })

    return pd.DataFrame(rows)


def compute_corpus_egs(results):
    contract_df = pd.DataFrame(
        [flatten_result_for_table(r) for r in results]
    )

    total_contracts = len(results)
    n_criteria = len(DECISION_CRITERIA)
    total_pairs = total_contracts * n_criteria
    grounded_pairs = int(contract_df["S"].sum()) if total_contracts else 0
    corpus_egs = grounded_pairs / total_pairs if total_pairs else 0.0

    scores = (
        contract_df["evidence_grounding_score"]
        if total_contracts
        else pd.Series(dtype=float)
    )

    return pd.DataFrame([{
        "assessment_dimension": ASSESSMENT_DIMENSION,
        "n_contracts": total_contracts,
        "n_criteria": n_criteria,
        "contract_criterion_pairs": total_pairs,
        "grounded_pairs": grounded_pairs,
        "corpus_evidence_grounding_score": round(corpus_egs, 4),
        "mean_contract_egs": (
            round(float(scores.mean()), 4) if total_contracts else 0.0
        ),
        "median_contract_egs": (
            round(float(scores.median()), 4) if total_contracts else 0.0
        ),
        "std_contract_egs": (
            round(float(scores.std(ddof=1)), 4)
            if total_contracts > 1
            else 0.0
        ),
        "min_contract_egs": (
            round(float(scores.min()), 4) if total_contracts else 0.0
        ),
        "max_contract_egs": (
            round(float(scores.max()), 4) if total_contracts else 0.0
        ),
    }])


def run_evaluation_batch(attachments, limit=None, sleep_seconds: float = 0.0):
    selected = list(attachments)
    if limit is not None:
        selected = selected[:int(limit)]

    results = []
    total = len(selected)

    for idx, attachment in enumerate(selected, start=1):
        print("=" * 120)
        print(f"[Phase 1 {idx}/{total}] {attachment.filename}")

        result = evaluate_contract(
            attachment,
            sleep_seconds=sleep_seconds,
        )
        results.append(result)

        print(
            f"  EGS={result['evidence_grounding_score']} | "
            f"Decision={result['final_decision']} | "
            f"States={result['state_counts']}"
        )
        print()

    return results


def save_phase1_results(results):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    json_path = EVALUATION_OUTPUT_DIR / (
        f"decision_batch_results_{timestamp}.json"
    )
    contract_csv_path = EVALUATION_OUTPUT_DIR / (
        f"decision_batch_summary_{timestamp}.csv"
    )
    criterion_csv_path = EVALUATION_OUTPUT_DIR / (
        f"egs_by_criterion_{timestamp}.csv"
    )
    corpus_csv_path = EVALUATION_OUTPUT_DIR / (
        f"egs_corpus_summary_{timestamp}.csv"
    )

    json_path.write_text(
        json.dumps(results, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    contract_df = pd.DataFrame(
        [flatten_result_for_table(r) for r in results]
    )
    criterion_df = compute_criterion_egs(results)
    corpus_df = compute_corpus_egs(results)

    contract_df.to_csv(
        contract_csv_path,
        index=False,
        encoding="utf-8-sig",
    )
    criterion_df.to_csv(
        criterion_csv_path,
        index=False,
        encoding="utf-8-sig",
    )
    corpus_df.to_csv(
        corpus_csv_path,
        index=False,
        encoding="utf-8-sig",
    )

    return (
        json_path,
        contract_csv_path,
        criterion_csv_path,
        corpus_csv_path,
        contract_df,
        criterion_df,
        corpus_df,
    )


print("Assessment dimension discovered from bridge:", ASSESSMENT_DIMENSION)
print("Decision criteria discovered from bridge:", len(DECISION_CRITERIA))
for c in DECISION_CRITERIA:
    print(
        "-", c["criterion_id"],
        "| target:", c["target"],
        "| requirements:", c["signature_requirements"],
    )
print("Decision levels discovered from bridge:", DECISION_LEVELS)
print("Decision rules discovered from bridge:", DECISION_LEVEL_RULES)


In [ ]:
# ==============================
# Run Phase 1
# ==============================
phase1_results = run_evaluation_batch(
    ATTACHMENTS,
    limit=BATCH_LIMIT,
    sleep_seconds=SLEEP_SECONDS,
)

(
    phase1_json_path,
    phase1_contract_csv_path,
    phase1_criterion_csv_path,
    phase1_corpus_csv_path,
    phase1_contract_df,
    phase1_criterion_egs_df,
    phase1_corpus_egs_df,
) = save_phase1_results(phase1_results)

print("Generated files:")
print("-", phase1_json_path)
print("-", phase1_contract_csv_path)
print("-", phase1_criterion_csv_path)
print("-", phase1_corpus_csv_path)

print("\nContract-level summary:")
display(phase1_contract_df)

print("\nEGS by criterion:")
display(phase1_criterion_egs_df)

print("\nCorpus-level EGS:")
display(phase1_corpus_egs_df)


## Phase 2 — Neural Assessment of Non-Verifiable Conditions

The second phase is executed **only for criteria whose formal state is `P` or `U`**.

States `S` and `C` are already considered conclusively determined by the explicit organizational evidence:

- `S`: the truthmaker conditions are formally established;
- `C`: explicit organizational evidence establishes that the condition does not hold.

For `P` and `U`, the neural component may return:

- `Supported`: semantic interpretation of the available evidence provides sufficient complementary support;
- `Contradicted`: semantic interpretation provides sufficient complementary contradiction;
- `Still Undetermined`: the available evidence remains insufficient.

The neural assessment never changes the original formal state. It is stored separately so that epistemic provenance is preserved.


In [ ]:
# ==============================
# DGS configuration
# ==============================

NEURAL_ASSESSMENT_VALUES = {
    "Supported",
    "Contradicted",
    "Still Undetermined",
}


def compute_symbolic_dgs(conditions):
    """
    DGS_symbolic(d) = (#S + #C) / |C|
    """
    conditions = normalize_decision_criteria(conditions)
    determined = sum(
        1 for c in conditions
        if c["state"] in {"S", "C"}
    )
    denominator = len(DECISION_CRITERIA)
    return round(
        determined / denominator if denominator else 0.0,
        4,
    )


def _normalize_neural_assessment(value):
    raw = str(value or "").strip().lower()

    mapping = {
        "supported": "Supported",
        "support": "Supported",
        "satisfied": "Supported",
        "yes": "Supported",
        "contradicted": "Contradicted",
        "contradiction": "Contradicted",
        "not supported": "Contradicted",
        "no": "Contradicted",
        "still undetermined": "Still Undetermined",
        "undetermined": "Still Undetermined",
        "unresolved": "Still Undetermined",
    }

    return mapping.get(raw, "Still Undetermined")


def compute_neuro_symbolic_dgs(conditions, neural_assessments):
    conditions = normalize_decision_criteria(conditions)

    neural_by_id = {
        str(a.get("criterion_id", "")).strip().lower(): a
        for a in (neural_assessments or [])
    }

    determined = 0

    for cond in conditions:
        cid = cond["criterion_id"]
        state = cond["state"]

        if state in {"S", "C"}:
            determined += 1
            continue

        if state in {"P", "U"}:
            assessment = _normalize_neural_assessment(
                neural_by_id.get(cid, {}).get("assessment")
            )
            if assessment in {"Supported", "Contradicted"}:
                determined += 1

    denominator = len(DECISION_CRITERIA)
    return round(
        determined / denominator if denominator else 0.0,
        4,
    )


def effective_positive_criteria(conditions, neural_assessments):
    """
    Positive criteria used only for the complementary neuro-symbolic
    recommendation.

    - Formal S always counts.
    - Formal C never counts.
    - P/U count only when neural assessment == Supported.
    """
    conditions = normalize_decision_criteria(conditions)

    neural_by_id = {
        str(a.get("criterion_id", "")).strip().lower(): a
        for a in (neural_assessments or [])
    }

    positive = set()

    for cond in conditions:
        cid = cond["criterion_id"]
        state = cond["state"]

        if state == "S":
            positive.add(cid)
        elif state in {"P", "U"}:
            assessment = _normalize_neural_assessment(
                neural_by_id.get(cid, {}).get("assessment")
            )
            if assessment == "Supported":
                positive.add(cid)

    return sorted(positive)


In [ ]:
def build_neural_complement_prompt(
    attachment: AttachmentText,
    condition: dict,
    phase1_result: dict,
) -> str:
    """
    Builds a narrowly scoped prompt for one unresolved criterion.

    Criterion semantics are obtained from the Semantic Bridge Model rather
    than embedded in the prompt.
    """
    cid = condition["criterion_id"]
    descriptor = DECISION_CRITERION_BY_ID[cid]
    state = condition["state"]

    descriptor_json = json.dumps(
        descriptor,
        ensure_ascii=False,
        indent=2,
    )

    return f"""
You are performing a SECOND-STAGE NEURAL ASSESSMENT for one unresolved
decision criterion defined by the Decision Domain.

The formal evidence-grounding assessment has already been completed.

Epistemic constraints:
1. Do not change or reinterpret criteria already formally established as S or C.
2. Evaluate only criterion {cid}.
3. Do not infer or return the final decision level.
4. Do not invent contract-specific facts.
5. General domain knowledge may be used only to interpret the meaning and
   implications of evidence that is actually present in the supplied
   organizational artifacts.
6. The Semantic Bridge Model defines the criterion semantics; it is not
   contract-specific evidence.
7. If the available organizational evidence still does not justify a more
   conclusive assessment, return "Still Undetermined".
8. "Supported" means that semantic interpretation of the available evidence
   provides sufficient complementary support for the criterion.
9. "Contradicted" means that semantic interpretation of the available evidence
   provides sufficient complementary support for the criterion not holding.
10. This neural result is complementary. It does not create or overwrite a
    formal truthmaker and does not change the original S/P/U/C state.

Criterion definition parsed from the Semantic Bridge Model:
---
{descriptor_json}
---

Previous formal state:
{state}

Phase-1 evidence:
{condition.get("evidence", "")}

Phase-1 rationale:
{condition.get("rationale", "")}

Return STRICT JSON only:

{{
  "criterion_id": "{cid}",
  "formal_state": "{state}",
  "assessment": "Supported|Contradicted|Still Undetermined",
  "evidence_used": "short excerpt or paraphrase from the organizational artifacts",
  "neural_rationale": "brief explanation of the semantic interpretation",
  "epistemic_note": "why this result is complementary rather than formal"
}}

SEMANTIC BRIDGE MODEL:
---
{SEMANTIC_BRIDGE_PL}
---

ORGANIZATIONAL EVIDENCE:
---
{render_contract_context(attachment)}
---

Return JSON only.
""".strip()


def evaluate_unresolved_criterion_neurally(
    attachment: AttachmentText,
    condition: dict,
    phase1_result: dict,
    sleep_seconds: float = 0.0,
):
    if condition["state"] not in {"P", "U"}:
        raise ValueError(
            "Neural complementation is allowed only for formal P/U criteria."
        )

    prompt = build_neural_complement_prompt(
        attachment=attachment,
        condition=condition,
        phase1_result=phase1_result,
    )

    response = client.models.generate_content(
        model=model_name,
        contents=prompt,
    )
    raw_text = getattr(response, "text", "").strip()
    parsed = _extract_json_object(raw_text)

    if parsed is None:
        parsed = {
            "criterion_id": condition["criterion_id"],
            "formal_state": condition["state"],
            "assessment": "Still Undetermined",
            "evidence_used": "",
            "neural_rationale": (
                "The neural complement could not be parsed as structured JSON."
            ),
            "epistemic_note": (
                "No complementary determination was recorded because the "
                "neural output was not structurally usable."
            ),
            "parse_error": True,
            "raw_response": raw_text,
        }
    else:
        parsed["parse_error"] = False
        parsed["raw_response"] = raw_text

    parsed["criterion_id"] = condition["criterion_id"]
    parsed["formal_state"] = condition["state"]
    parsed["assessment"] = _normalize_neural_assessment(
        parsed.get("assessment")
    )

    if sleep_seconds and sleep_seconds > 0:
        time.sleep(sleep_seconds)

    return parsed


In [ ]:
def run_neural_complement_for_contract(
    attachment: AttachmentText,
    phase1_result: dict,
    sleep_seconds: float = 0.0,
):
    conditions = normalize_decision_criteria(
        phase1_result.get("conditions", [])
    )

    neural_assessments = []

    for cond in conditions:
        if cond["state"] not in {"P", "U"}:
            continue

        print(
            f"  Neural complement: {cond['criterion_id']} "
            f"({cond['state']})"
        )

        assessment = evaluate_unresolved_criterion_neurally(
            attachment=attachment,
            condition=cond,
            phase1_result=phase1_result,
            sleep_seconds=sleep_seconds,
        )
        neural_assessments.append(assessment)

    symbolic_dgs = compute_symbolic_dgs(conditions)
    neuro_dgs = compute_neuro_symbolic_dgs(
        conditions,
        neural_assessments,
    )
    delta_dgs = round(neuro_dgs - symbolic_dgs, 4)

    positive_criteria = effective_positive_criteria(
        conditions,
        neural_assessments,
    )

    recommendation = derive_decision_from_positive_criteria(
        positive_criteria
    )

    return {
        "contract_file": phase1_result.get("contract_file"),
        "contract_label": phase1_result.get("contract_label"),
        "assessment_dimension": ASSESSMENT_DIMENSION,
        "formal_conditions": conditions,
        "neural_assessments": neural_assessments,
        "evidence_grounding_score": phase1_result.get(
            "evidence_grounding_score"
        ),
        "symbolic_decision": phase1_result.get("final_decision"),
        "symbolic_dgs": symbolic_dgs,
        "neuro_symbolic_dgs": neuro_dgs,
        "delta_dgs": delta_dgs,
        "neuro_symbolic_recommendation": recommendation["decision_label"],
        "neuro_symbolic_inference": {
            "positive_criteria": recommendation["positive_criteria"],
            "matched_levels": recommendation["matched_levels"],
            "winning_rule": recommendation["winning_rule"],
            "symbolic_level": recommendation["symbolic_level"],
        },
    }


def run_neural_complement_batch(
    attachments,
    phase1_results,
    limit=None,
    sleep_seconds: float = 0.0,
):
    attachment_by_filename = {
        a.filename: a for a in attachments
    }

    selected_results = list(phase1_results)
    if limit is not None:
        selected_results = selected_results[:int(limit)]

    outputs = []
    total = len(selected_results)

    for idx, phase1_result in enumerate(selected_results, start=1):
        filename = phase1_result.get("contract_file")
        attachment = attachment_by_filename.get(filename)

        if attachment is None:
            raise KeyError(
                f"Attachment not found for Phase-1 result: {filename}"
            )

        print("=" * 120)
        print(
            f"[Phase 2 {idx}/{total}] {filename} | "
            f"EGS={phase1_result.get('evidence_grounding_score')} | "
            f"Symbolic={phase1_result.get('final_decision')}"
        )

        result = run_neural_complement_for_contract(
            attachment=attachment,
            phase1_result=phase1_result,
            sleep_seconds=sleep_seconds,
        )
        outputs.append(result)

        print(
            f"  DGS symbolic={result['symbolic_dgs']} | "
            f"DGS neuro-symbolic={result['neuro_symbolic_dgs']} | "
            f"Delta DGS={result['delta_dgs']} | "
            f"Recommendation={result['neuro_symbolic_recommendation']}"
        )
        print()

    return outputs


### Phase 2 execution configuration

Neural complementation may require multiple model calls because each `P` or `U` criterion is assessed independently.

For a quick test, set `NEURAL_BATCH_LIMIT` to a small integer. Set it to `None` to process the complete corpus.


In [ ]:
# ==============================
# Run Phase 2
# ==============================

NEURAL_BATCH_LIMIT = None
NEURAL_SLEEP_SECONDS = SLEEP_SECONDS

neural_complement_results = run_neural_complement_batch(
    ATTACHMENTS,
    phase1_results,
    limit=NEURAL_BATCH_LIMIT,
    sleep_seconds=NEURAL_SLEEP_SECONDS,
)


In [ ]:
def flatten_neural_complement_result(result):
    assessments = result.get("neural_assessments", []) or []

    n_supported = sum(
        1 for a in assessments
        if a.get("assessment") == "Supported"
    )
    n_contradicted = sum(
        1 for a in assessments
        if a.get("assessment") == "Contradicted"
    )
    n_still_undetermined = sum(
        1 for a in assessments
        if a.get("assessment") == "Still Undetermined"
    )

    return {
        "contract_file": result.get("contract_file"),
        "contract_label": result.get("contract_label"),
        "assessment_dimension": result.get("assessment_dimension"),
        "egs": result.get("evidence_grounding_score"),
        "symbolic_dgs": result.get("symbolic_dgs"),
        "neuro_symbolic_dgs": result.get("neuro_symbolic_dgs"),
        "delta_dgs": result.get("delta_dgs"),
        "symbolic_decision": result.get("symbolic_decision"),
        "neuro_symbolic_recommendation": result.get(
            "neuro_symbolic_recommendation"
        ),
        "neural_supported": n_supported,
        "neural_contradicted": n_contradicted,
        "neural_still_undetermined": n_still_undetermined,
        "neural_assessments_count": len(assessments),
    }


neural_complement_df = pd.DataFrame(
    [
        flatten_neural_complement_result(r)
        for r in neural_complement_results
    ]
)

display(neural_complement_df)


In [ ]:
def compute_dgs_corpus_summary(results):
    df = pd.DataFrame(
        [
            flatten_neural_complement_result(r)
            for r in results
        ]
    )

    if df.empty:
        return pd.DataFrame()

    n_contracts = len(df)
    n_criteria = len(DECISION_CRITERIA)
    n_pairs = n_contracts * n_criteria

    symbolic_determined_pairs = round(
        float(df["symbolic_dgs"].sum()) * n_criteria
    )
    neuro_determined_pairs = round(
        float(df["neuro_symbolic_dgs"].sum()) * n_criteria
    )

    return pd.DataFrame([{
        "assessment_dimension": ASSESSMENT_DIMENSION,
        "n_contracts": n_contracts,
        "n_criteria": n_criteria,
        "contract_criterion_pairs": n_pairs,
        "symbolic_determined_pairs": symbolic_determined_pairs,
        "corpus_symbolic_dgs": round(
            symbolic_determined_pairs / n_pairs,
            4,
        ),
        "neuro_symbolic_determined_pairs": neuro_determined_pairs,
        "corpus_neuro_symbolic_dgs": round(
            neuro_determined_pairs / n_pairs,
            4,
        ),
        "additional_pairs_determined_by_neural": (
            neuro_determined_pairs - symbolic_determined_pairs
        ),
        "corpus_delta_dgs": round(
            (neuro_determined_pairs - symbolic_determined_pairs) / n_pairs,
            4,
        ),
        "mean_contract_symbolic_dgs": round(
            float(df["symbolic_dgs"].mean()),
            4,
        ),
        "mean_contract_neuro_symbolic_dgs": round(
            float(df["neuro_symbolic_dgs"].mean()),
            4,
        ),
        "mean_contract_delta_dgs": round(
            float(df["delta_dgs"].mean()),
            4,
        ),
        "median_neuro_symbolic_dgs": round(
            float(df["neuro_symbolic_dgs"].median()),
            4,
        ),
        "std_neuro_symbolic_dgs": (
            round(float(df["neuro_symbolic_dgs"].std(ddof=1)), 4)
            if len(df) > 1
            else 0.0
        ),
    }])


dgs_corpus_df = compute_dgs_corpus_summary(
    neural_complement_results
)
display(dgs_corpus_df)


In [ ]:
def compute_dgs_by_criterion(results):
    rows = []

    for criterion in DECISION_CRITERIA:
        cid = criterion["criterion_id"]

        formal_determined = 0
        neuro_determined = 0
        neural_supported = 0
        neural_contradicted = 0
        still_undetermined = 0

        for result in results:
            cond = next(
                c for c in result["formal_conditions"]
                if c["criterion_id"] == cid
            )

            if cond["state"] in {"S", "C"}:
                formal_determined += 1
                neuro_determined += 1
                continue

            assessment = next(
                (
                    a for a in result.get("neural_assessments", [])
                    if a.get("criterion_id") == cid
                ),
                None,
            )

            neural_value = _normalize_neural_assessment(
                (assessment or {}).get("assessment")
            )

            if neural_value == "Supported":
                neural_supported += 1
                neuro_determined += 1
            elif neural_value == "Contradicted":
                neural_contradicted += 1
                neuro_determined += 1
            else:
                still_undetermined += 1

        total = len(results)

        rows.append({
            "criterion_id": cid,
            "criterion_label": criterion["criterion_label"],
            "criterion_target": criterion["target"],
            "formal_determined": formal_determined,
            "symbolic_dgs": (
                round(formal_determined / total, 4)
                if total else 0.0
            ),
            "neuro_symbolic_determined": neuro_determined,
            "neuro_symbolic_dgs": (
                round(neuro_determined / total, 4)
                if total else 0.0
            ),
            "delta_dgs": (
                round(
                    (neuro_determined - formal_determined) / total,
                    4,
                )
                if total else 0.0
            ),
            "neural_supported": neural_supported,
            "neural_contradicted": neural_contradicted,
            "still_undetermined_after_neural": still_undetermined,
        })

    return pd.DataFrame(rows)


dgs_by_criterion_df = compute_dgs_by_criterion(
    neural_complement_results
)
display(dgs_by_criterion_df)


In [ ]:
def compute_recommendation_distribution(results):
    total = len(results)
    rows = []

    for level in DECISION_LEVELS:
        label = DECISION_LEVEL_LABELS[level]

        n_symbolic = sum(
            1 for r in results
            if r.get("symbolic_decision") == label
        )
        n_neuro = sum(
            1 for r in results
            if r.get("neuro_symbolic_recommendation") == label
        )

        rows.append({
            "decision_level": level,
            "decision_label": label,
            "symbolic_contracts": n_symbolic,
            "symbolic_proportion": (
                round(n_symbolic / total, 4)
                if total else 0.0
            ),
            "neuro_symbolic_contracts": n_neuro,
            "neuro_symbolic_proportion": (
                round(n_neuro / total, 4)
                if total else 0.0
            ),
        })

    return pd.DataFrame(rows)


recommendation_distribution_df = compute_recommendation_distribution(
    neural_complement_results
)
display(recommendation_distribution_df)

changed_recommendations_df = neural_complement_df[
    neural_complement_df["symbolic_decision"]
    != neural_complement_df["neuro_symbolic_recommendation"]
].copy()

print(
    "Contracts whose complementary recommendation differs from "
    "the formal symbolic decision:",
    len(changed_recommendations_df),
)
display(changed_recommendations_df)


In [ ]:
# ==============================
# Save Phase 2 outputs
# ==============================

timestamp_phase2 = datetime.now().strftime("%Y%m%d_%H%M%S")

phase2_json_path = EVALUATION_OUTPUT_DIR / (
    f"decision_neural_complement_results_{timestamp_phase2}.json"
)
phase2_contract_csv_path = EVALUATION_OUTPUT_DIR / (
    f"decision_dgs_contracts_{timestamp_phase2}.csv"
)
phase2_corpus_csv_path = EVALUATION_OUTPUT_DIR / (
    f"decision_dgs_corpus_{timestamp_phase2}.csv"
)
phase2_criterion_csv_path = EVALUATION_OUTPUT_DIR / (
    f"decision_dgs_by_criterion_{timestamp_phase2}.csv"
)
phase2_distribution_csv_path = EVALUATION_OUTPUT_DIR / (
    f"decision_symbolic_vs_neurosymbolic_{timestamp_phase2}.csv"
)

phase2_json_path.write_text(
    json.dumps(
        neural_complement_results,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

neural_complement_df.to_csv(
    phase2_contract_csv_path,
    index=False,
    encoding="utf-8-sig",
)
dgs_corpus_df.to_csv(
    phase2_corpus_csv_path,
    index=False,
    encoding="utf-8-sig",
)
dgs_by_criterion_df.to_csv(
    phase2_criterion_csv_path,
    index=False,
    encoding="utf-8-sig",
)
recommendation_distribution_df.to_csv(
    phase2_distribution_csv_path,
    index=False,
    encoding="utf-8-sig",
)

print("Phase-2 files generated:")
print("-", phase2_json_path)
print("-", phase2_contract_csv_path)
print("-", phase2_corpus_csv_path)
print("-", phase2_criterion_csv_path)
print("-", phase2_distribution_csv_path)


## Metric interpretation

### Evidence Grounding Score (EGS)

\[
EGS(d)=\frac{\#S}{|C|}
\]

EGS measures **positive formal grounding**. Neural complementation never changes EGS.

### Symbolic Decision Grounding Score

\[
DGS_{\text{symbolic}}(d)=
\frac{\#S+\#C}{|C|}
\]

This metric measures the fraction of decision criteria formally determined, either positively or negatively.

### Neuro-Symbolic Decision Grounding Score

\[
DGS_{\text{NS}}(d)=
\frac{
\#(S,C)
+
\#(\text{P/U resolved by neural assessment})
}{|C|}
\]

This metric measures how much of the criterion set can be conclusively assessed after neural complementation.

### Neural Gain

\[
\Delta DGS(d)=
DGS_{\text{NS}}(d)-DGS_{\text{symbolic}}(d)
\]

This measures the additional criterion determination attributable to neural complementation.

### Decision provenance

The output preserves two distinct results:

- `symbolic_decision`: the formal conclusion derived only from criteria in state `S`;
- `neuro_symbolic_recommendation`: a complementary result derived from formal `S` criteria plus `P/U` criteria assessed as `Supported`.

The complementary result never overwrites the formal conclusion. This separation makes the epistemic basis of the final explanation inspectable.


## Evidence Grounding Score interpretation

The `Evidence Grounding Score (EGS)` measures the proportion of Decision Domain criteria whose truthmaker conditions are established by the available organizational evidence.

- `S` (Satisfied) contributes `1` to formal grounding.
- `P`, `U`, and `C` contribute `0` to EGS, while their epistemic states remain available for analysis.
- `EGS = 1.00` means that every criterion defined for the selected assessment dimension is formally grounded.
- Intermediate values mean that only part of the criterion set has formal positive support.
- `EGS = 0.00` means that none of the criteria could be formally established from the available organizational evidence.

A low EGS does not mean that the criteria are false or that the decision is incorrect. It means only that the available organizational evidence does not formally establish a larger proportion of the conditions required by the Decision Domain.

The formal decision is independent of EGS. It is derived by applying the `level_rule/3` facts from the Semantic Bridge Model to criteria in state `S`.
